# Extract Kikuchi Lines

In [ ]:
from pathlib import Path

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi
from skimage import draw, exposure, filters, morphology, transform

plt.rcParams["figure.figsize"] = (12, 6)

pattern_path = Path("patterns/1.png")
edge_margin = 8
background_sigma = 9.0
line_quantile = 0.80
min_object_size = 25
hough_threshold = 5
hough_line_length = 22
hough_line_gap = 6
min_segment_length = 28.0


In [ ]:
img = imageio.imread(pattern_path)
if img.ndim == 3:
    gray = img[..., :3].mean(axis=2).astype(np.float32)
    mask = img[..., 3] > 0 if img.shape[2] == 4 else np.ones(img.shape[:2], dtype=bool)
else:
    gray = img.astype(np.float32)
    mask = gray > 0

mask = ndi.binary_erosion(mask, iterations=edge_margin)
gray = exposure.rescale_intensity(gray, in_range="image", out_range=(0.0, 1.0))
background = filters.gaussian(gray, sigma=background_sigma)
corrected = gray - background
corrected = exposure.rescale_intensity(corrected, in_range="image", out_range=(0.0, 1.0))
line_response = filters.meijering(corrected, sigmas=range(1, 6), black_ridges=False)
line_response = exposure.rescale_intensity(line_response, in_range="image", out_range=(0.0, 1.0))

threshold = np.quantile(line_response[mask], line_quantile)
binary = (line_response > threshold) & mask
binary = morphology.remove_small_objects(binary, min_size=min_object_size)
binary = morphology.binary_closing(binary, morphology.disk(1))

raw_lines = transform.probabilistic_hough_line(
    binary,
    threshold=hough_threshold,
    line_length=hough_line_length,
    line_gap=hough_line_gap,
)

line_only = np.zeros_like(gray, dtype=np.uint8)
lines = []
for (x0, y0), (x1, y1) in raw_lines:
    length = float(np.hypot(x1 - x0, y1 - y0))
    if length < min_segment_length:
        continue
    rr, cc = draw.line(y0, x0, y1, x1)
    line_only[rr, cc] = 255
    lines.append(((x0, y0), (x1, y1)))

print(f"Pattern: {pattern_path}")
print(f"Kept line segments: {len(lines)}")
print(f"Threshold: {threshold:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(gray, cmap="gray")
axes[0].imshow(line_only, cmap="autumn", alpha=0.95)
axes[0].set_title("Overlay")
axes[0].axis("off")

axes[1].imshow(line_only, cmap="gray")
axes[1].set_title("Line Only")
axes[1].axis("off")

plt.tight_layout()
plt.show()
